# Retrieval-Augmented Generation (RAG) with LangChain

## What is RAG?
**Retrieval-Augmented Generation (RAG)** is a technique that enhances Large Language Models (LLMs) by providing them with relevant external knowledge. Instead of relying solely on the model's training data, RAG retrieves relevant documents from a knowledge base and uses them as context for generating responses.

### Why RAG?
- **Reduces hallucinations**: LLMs can make up facts; RAG grounds responses in actual documents
- **Up-to-date information**: Knowledge bases can be updated without retraining the model
- **Domain-specific knowledge**: Easily add custom documents for specialized use cases
- **Transparent sources**: You can trace where the information came from

### RAG Pipeline Overview:
1. **Load Documents** → Load your data from various sources (text, PDF, web, etc.)
2. **Split Documents** → Break documents into smaller chunks for better retrieval
3. **Create Embeddings** → Convert text chunks into numerical vectors
4. **Store in Vector DB** → Save embeddings in a vector database for fast similarity search
5. **Retrieve** → Find the most relevant chunks for a user query
6. **Generate** → Pass retrieved context to LLM to generate a response

---
Link: https://www.udemy.com/course/langchain-in-action-develop-llm-powered-applications/

## Step 1: Document Loaders

Document Loaders are the entry point for bringing external data into LangChain. They handle reading data from various sources and converting it into a standard `Document` format.

### Supported Loaders:
- **TextLoader**: Plain text files (.txt)
- **PyPDFLoader**: PDF documents
- **CSVLoader**: CSV files
- **WebBaseLoader**: Web pages
- **DirectoryLoader**: Multiple files from a directory
- And many more...

### Document Structure:
Each document has two main components:
- `page_content`: The actual text content
- `metadata`: Additional information (source, page number, etc.)

In [ ]:
# =============================================================================
# Environment Setup
# =============================================================================
# Load environment variables from .env file
# This is where we store sensitive information like API keys
# Your .env file should contain: OPENAI_API_KEY=your-api-key-here

from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

In [ ]:
# =============================================================================
# Loading Documents with TextLoader
# =============================================================================
# TextLoader reads a plain text file and converts it into a LangChain Document
# The document contains:
#   - page_content: The full text content of the file
#   - metadata: Information about the document (e.g., source file path)

from langchain_community.document_loaders import TextLoader

# Initialize the loader with the path to your text file
loader = TextLoader("./bella_vista.txt")

# load() method reads the file and returns a list of Document objects
# For a single text file, this returns a list with one Document
docs = loader.load()

In [ ]:
# Examine the loaded documents
# Notice that the entire file is loaded as a single Document
# This is important - we'll need to split it into smaller chunks later!

print(docs)
print(f"Number of documents loaded: {len(docs)}")

In [ ]:
# =============================================================================
# Creating Documents Manually
# =============================================================================
# You can also create Document objects programmatically
# This is useful when you have data from other sources (databases, APIs, etc.)

from langchain.schema import Document

# Create a Document with custom content and metadata
# Metadata can store any key-value pairs you need (source, author, date, etc.)
example_doc = Document(
    page_content="test",  # The actual text content
    metadata={"important_info": "hi there"}  # Additional context/info
)
example_doc

## Step 2: Text Splitting (Chunking)

### Why Split Documents?
Large documents cannot be efficiently processed by LLMs or stored in vector databases. We need to split them into smaller, manageable pieces called **chunks**.

### Key Concepts:
- **Chunk Size**: Maximum number of characters in each chunk
- **Chunk Overlap**: Number of characters shared between consecutive chunks (preserves context at boundaries)

### Types of Text Splitters:
- **RecursiveCharacterTextSplitter**: Splits by characters, trying to keep paragraphs/sentences together (RECOMMENDED)
- **CharacterTextSplitter**: Simple split by a separator
- **TokenTextSplitter**: Splits by tokens (useful for token-limited models)
- **MarkdownTextSplitter**: Respects markdown structure

### Best Practices:
- Chunk size depends on your use case (typically 500-1500 characters)
- Overlap helps maintain context (typically 10-20% of chunk size)
- Consider the semantic meaning when choosing split points

In [ ]:
# =============================================================================
# Splitting Documents into Chunks
# =============================================================================
# RecursiveCharacterTextSplitter is the recommended splitter for most use cases
# It tries to split text at natural boundaries (paragraphs, sentences, words)
# in that order, until chunks are small enough

from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,      # Maximum characters per chunk (small for demo purposes)
    chunk_overlap=20,    # Characters shared between consecutive chunks
    # Default separators: ["\n\n", "\n", " ", ""] (paragraphs → lines → words → chars)
)

# Split the loaded documents into smaller chunks
# Each chunk becomes its own Document with inherited metadata
documents = text_splitter.split_documents(docs)

In [ ]:
# View all the chunks created from our document
# Notice how the original document is now split into multiple smaller pieces
# Each chunk maintains the original metadata (source file)

documents

In [ ]:
# =============================================================================
# Examining Individual Chunks
# =============================================================================
# Let's look at each chunk in detail
# Notice how overlap works - some text appears at the end of one chunk
# and the beginning of the next

for i, doc in enumerate(documents):
    print(f"--- Chunk {i+1} ---")
    print(f"Content: {doc.page_content}")
    print(f"Metadata: {doc.metadata}")
    print()

print(f"Total number of chunks: {len(documents)}")

## Step 3: Embeddings

### What are Embeddings?
**Embeddings** are numerical representations (vectors) of text that capture semantic meaning. Similar texts have similar embeddings, allowing us to find related content through mathematical operations.

### How Embeddings Work:
- Text → Embedding Model → Vector (list of numbers)
- Example: "Hello" → [0.023, -0.045, 0.089, ...] (typically 1536 dimensions for OpenAI)

### Why Embeddings Matter for RAG:
- Enable **semantic search**: Find documents by meaning, not just keywords
- Allow **similarity comparison**: Measure how related two texts are
- Power **vector databases**: Store and query documents efficiently

### Embedding Models:
- **OpenAI Embeddings**: High quality, requires API key (text-embedding-ada-002, text-embedding-3-small)
- **HuggingFace Embeddings**: Open source alternatives
- **Cohere, Google, etc.**: Various cloud providers

### Key Concept: The embedding model used for storing MUST match the one used for querying!

In [ ]:
# =============================================================================
# Initialize OpenAI Embeddings
# =============================================================================
# OpenAIEmbeddings uses OpenAI's embedding models to convert text to vectors
# Default model: text-embedding-ada-002 (1536 dimensions)
# Newer option: text-embedding-3-small or text-embedding-3-large

from langchain_openai import OpenAIEmbeddings

# Initialize the embeddings model
# Uses OPENAI_API_KEY from environment variables automatically
embeddings = OpenAIEmbeddings()

# You can also specify a specific model:
# embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
# =============================================================================
# Creating Embeddings for Text
# =============================================================================
# embed_query() converts a single text string into a vector
# The resulting vector is a list of floating-point numbers

embedding1 = embeddings.embed_query(
    text="The solar system consists of the Sun and the objects that orbit it"
)

# Display the embedding vector (truncated for readability)
print(f"First 10 dimensions: {embedding1[:10]}")
print(f"Total dimensions: {len(embedding1)}")
print(f"Data type of each dimension: {type(embedding1[0])}")

In [ ]:
# =============================================================================
# Creating Multiple Embeddings for Comparison
# =============================================================================
# Let's create embeddings for different texts to demonstrate semantic similarity
# We'll compare:
#   - embedding2: Same text as embedding1 (should be identical)
#   - embedding3: Related topic (solar system) - should be similar
#   - embedding4: Unrelated topic (baking) - should be dissimilar

embedding2 = embeddings.embed_query(
    text="The solar system consists of the Sun and the objects that orbit it"  # Same as embedding1
)
embedding3 = embeddings.embed_query(
    text="Planets, asteroids, and comets are part of our solar system."  # Related topic
)
embedding4 = embeddings.embed_query(
    text="I love baking chocolate chip cookies on weekends."  # Unrelated topic
)

In [ ]:
# =============================================================================
# Understanding Cosine Similarity
# =============================================================================
# Cosine similarity measures the angle between two vectors
# 
# Formula: cos(θ) = (A · B) / (||A|| × ||B||)
# 
# Values range from -1 to 1:
#   - 1.0  = Identical (vectors point in same direction)
#   - 0.0  = Orthogonal (no relationship)
#   - -1.0 = Opposite (vectors point in opposite directions)
# 
# For text embeddings, values typically range from 0.6 to 1.0
# Higher values = more semantically similar

import numpy as np

def cosine_similarity(A, B):
    """
    Calculate the cosine similarity between two vectors.
    
    Args:
        A: First embedding vector
        B: Second embedding vector
    
    Returns:
        Similarity score between -1 and 1
    """
    dot_product = np.dot(A, B)        # A · B
    norm_a = np.linalg.norm(A)         # ||A||
    norm_b = np.linalg.norm(B)         # ||B||
    return dot_product / (norm_a * norm_b)

In [ ]:
# =============================================================================
# Comparing Embedding Similarities
# =============================================================================
# Let's see how similar our different texts are!

sim_1_2 = cosine_similarity(embedding1, embedding2)  # Identical texts
sim_1_3 = cosine_similarity(embedding1, embedding3)  # Related topics
sim_3_4 = cosine_similarity(embedding3, embedding4)  # Unrelated topics

print("Similarity Comparisons:")
print("=" * 50)
print(f"Identical texts (solar system):     {sim_1_2:.4f}")
print(f"Related topics (solar system):     {sim_1_3:.4f}")
print(f"Unrelated topics (space vs baking): {sim_3_4:.4f}")
print()
print("Notice:")
print("- Identical texts have similarity ≈ 1.0")
print("- Related topics have high similarity (> 0.8)")
print("- Unrelated topics have lower similarity (< 0.8)")


## Step 4: Vector Database (FAISS)

### What is a Vector Database?
A **Vector Database** is a specialized database designed to store and query high-dimensional vectors (embeddings). It enables fast similarity search across millions of vectors.

### FAISS (Facebook AI Similarity Search):
- Open-source library from Meta/Facebook
- Highly efficient for similarity search
- Supports various indexing methods
- Can be stored locally (no external service needed)
- Great for prototyping and small-to-medium datasets

### Other Vector Databases:
| Database | Type | Best For |
|----------|------|----------|
| FAISS | Local | Prototyping, small datasets |
| Chroma | Local/Cloud | Easy setup, good for development |
| Pinecone | Cloud | Production, managed service |
| Weaviate | Cloud/Self-hosted | Hybrid search capabilities |
| Milvus | Self-hosted | Large-scale production |
| Qdrant | Cloud/Self-hosted | Production with filtering |

### Key Operations:
- **Insert**: Add documents and their embeddings
- **Search**: Find similar documents given a query
- **Filter**: Narrow results based on metadata

In [ ]:
# =============================================================================
# Creating and Saving a FAISS Vector Store
# =============================================================================
# FAISS.from_documents() does two things:
#   1. Creates embeddings for all document chunks
#   2. Indexes them in the FAISS vector store

from langchain.vectorstores.faiss import FAISS

# Create the vector store from our document chunks
# This automatically embeds all documents using the provided embedding model
vectorstore = FAISS.from_documents(
    documents,   # Our chunked documents
    embeddings   # The embedding model to use
)

# Save the vector store to disk for later use
# This creates two files: index.faiss (vectors) and index.pkl (metadata)
vectorstore.save_local("vs_db")

print("✅ Vector store created and saved to 'vs_db' directory")

## Step 5: Loading and Querying the Vector Store

### Loading a Saved Vector Store
Once saved, you can reload the vector store without re-embedding all documents. This is essential for production applications where you don't want to recreate embeddings every time.

### Security Note
The `allow_dangerous_deserialization=True` flag is required because pickle files can contain malicious code. Only load vector stores from trusted sources!

In [ ]:
# =============================================================================
# Loading a Previously Saved Vector Store
# =============================================================================
# Use load_local() to reload a saved vector store
# You must provide the same embedding model used during creation!

# Load from the "index" directory (a previously saved vector store)
loaded_vectorstore = FAISS.load_local(
    "index",                              # Directory containing saved files
    embeddings,                            # MUST match the original embedding model
    allow_dangerous_deserialization=True   # Required for pickle files - only use trusted sources!
)

print("✅ Vector store loaded successfully")
loaded_vectorstore

In [ ]:
# =============================================================================
# Creating a Retriever
# =============================================================================
# A Retriever is an interface for fetching relevant documents given a query
# It abstracts the vector store and provides a simple invoke() method
# 
# Benefits of using a Retriever:
# - Consistent interface across different vector stores
# - Easy to swap vector stores without changing code
# - Integrates seamlessly with LangChain chains

retriever = vectorstore.as_retriever()

# Default settings:
# - search_type: "similarity" (cosine similarity)
# - k: 4 (returns top 4 most similar documents)

In [ ]:
# =============================================================================
# Retrieving Relevant Documents
# =============================================================================
# Use invoke() to find documents most similar to your query
# The retriever automatically:
#   1. Embeds your query using the same embedding model
#   2. Performs similarity search in the vector store
#   3. Returns the top-k most similar documents

query = "When are the opening hours??"
docs = retriever.invoke(input=query)

print(f"Query: '{query}'")
print(f"Found {len(docs)} relevant documents:\n")

for i, doc in enumerate(docs, 1):
    print(f"--- Result {i} ---")
    print(f"Content: {doc.page_content}")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")
    print()


In [ ]:
# =============================================================================
# Retrieval with Filtering (Alternative Method)
# =============================================================================
# Note: The filter and k parameters in invoke() may not work for all vector stores
# For FAISS, use search_kwargs when creating the retriever (shown in next cell)

# This demonstrates the concept, but results depend on vector store implementation
docs = retriever.invoke(
    input="When are the opening hours?",
    filter={'source': './bella_vista.txt'},  # Filter by metadata
    k=3  # Number of results
)

print("Retrieved documents:")
for doc in docs:
    print(f"  - {doc.page_content[:80]}...")

In [ ]:
# =============================================================================
# Configuring Retriever with search_kwargs (RECOMMENDED)
# =============================================================================
# The proper way to configure retrieval parameters is through search_kwargs
# when creating the retriever

retriever = vectorstore.as_retriever(
    search_kwargs={
        "filter": {'source': './bella_vista.txt'},  # Only return docs from this source
        "k": 1  # Return only the top 1 most similar document
    }
)

# Now queries will use these settings automatically
docs = retriever.invoke(input="When are the opening hours??")

print("Configured retriever (k=1, filtered by source):")
for doc in docs:
    print(f"Content: {doc.page_content}")
    print(f"Metadata: {doc.metadata}")

## Step 6: Retrieval Chains - Combining Retrieval with LLM

### The RAG Pipeline Complete!
Now we connect all the pieces:
1. **User asks a question**
2. **Retriever finds relevant documents** (based on semantic similarity)
3. **Documents are passed as context to the LLM**
4. **LLM generates an answer** using the provided context

### Why This Works:
- LLM has access to relevant, up-to-date information
- Responses are grounded in your actual documents
- Reduces hallucination by providing factual context

### LangChain Retrieval Chains:
- **RetrievalQA** (deprecated): Simple question-answering chain
- **create_retrieval_chain** (new): Modern, flexible approach
- Both combine retrieval + LLM generation in one step

### Legacy Approach: RetrievalQA (Deprecated)

⚠️ **Note**: This approach still works but is deprecated. Use `create_retrieval_chain` (shown below) for new projects.

The legacy `RetrievalQA` chain uses:
- `chain_type="stuff"`: Stuffs all retrieved documents into the prompt (simple but limited by context window)

In [ ]:
# =============================================================================
# LEGACY: RetrievalQA Chain (Still works, but deprecated)
# =============================================================================
# This demonstrates the older pattern - use create_retrieval_chain for new code

from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# Create a custom prompt template
# The {context} placeholder will be filled with retrieved documents
# The {question} placeholder will be filled with the user's query
prompt_template = """You are a helpful assistant for our restaurant.

Context from our knowledge base:
{context}

Customer Question: {question}

Please provide a helpful answer based on the context above:"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]  # Variables that will be filled in
)

# Initialize the LLM (Chat model)
llm = ChatOpenAI(model="gpt-4o-mini")

# Create the RetrievalQA chain
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",  # "stuff" = put all documents into one prompt
    retriever=retriever,
    chain_type_kwargs={"prompt": PROMPT},
)

# Ask a question!
result = qa.invoke(input="When are the opening hours on sunday??")

print("=" * 50)
print("LEGACY RetrievalQA Result:")
print("=" * 50)
print(f"Query: {result['query']}")
print(f"Answer: {result['result']}")

### Modern Approach: create_retrieval_chain (Recommended) ✅

This is the **recommended approach** for new LangChain projects. It provides:
- Better composability with LCEL (LangChain Expression Language)
- More control over the retrieval and generation steps
- Cleaner separation of concerns

**Key Components:**
1. `create_stuff_documents_chain`: Combines documents into a single prompt
2. `create_retrieval_chain`: Orchestrates retrieval + document chain

In [ ]:
# =============================================================================
# MODERN: create_retrieval_chain (Recommended Approach)
# =============================================================================
# This is the current best practice for building RAG applications

from langchain_openai import ChatOpenAI
from langchain.chains.retrieval import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.prompts import PromptTemplate

# Create the prompt template
# NOTE: Use {input} instead of {question} for the new chain format
prompt_template = """You are a helpful and friendly assistant for Bella Vista restaurant.
Use the following context to answer the customer's question accurately.
If you don't know the answer based on the context, say so politely.

Context from our knowledge base:
{context}

Customer Question: {input}

Helpful Answer:"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "input"]  # Note: 'input' not 'question'
)

# Initialize the LLM
llm = ChatOpenAI(model="gpt-4o-mini")

# Step 1: Create a chain that combines documents into the prompt
# This chain takes documents and stuffs them into the {context} placeholder
combine_docs_chain = create_stuff_documents_chain(llm, PROMPT)

# Step 2: Create the full retrieval chain
# This chain: query → retriever → documents → combine_docs_chain → answer
qa = create_retrieval_chain(
    retriever=retriever,
    combine_docs_chain=combine_docs_chain
)

# Ask a question!
result = qa.invoke({"input": "When are the opening hours on sunday??"})

print("=" * 50)
print("MODERN create_retrieval_chain Result:")
print("=" * 50)
print(f"Question: {result['input']}")
print(f"\nRetrieved Context:")
for i, doc in enumerate(result['context'], 1):
    print(f"  {i}. {doc.page_content[:60]}...")
print(f"\n✨ Answer: {result['answer']}")

In [ ]:
# =============================================================================
# Summary: Complete RAG Pipeline
# =============================================================================
# 
# Congratulations! You've learned the complete RAG pipeline:
#
# 1. LOAD DOCUMENTS
#    - TextLoader, PDFLoader, etc.
#    - Convert files to Document objects
#
# 2. SPLIT INTO CHUNKS  
#    - RecursiveCharacterTextSplitter
#    - Choose appropriate chunk_size and chunk_overlap
#
# 3. CREATE EMBEDDINGS
#    - OpenAIEmbeddings (or other providers)
#    - Convert text to numerical vectors
#
# 4. STORE IN VECTOR DATABASE
#    - FAISS, Chroma, Pinecone, etc.
#    - Enable fast similarity search
#
# 5. CREATE RETRIEVER
#    - vectorstore.as_retriever()
#    - Configure k (number of results) and filters
#
# 6. BUILD RETRIEVAL CHAIN
#    - Combine retriever + LLM
#    - Use create_retrieval_chain (modern) or RetrievalQA (legacy)
#
# Next Steps:
# - Try different chunk sizes and see how it affects retrieval
# - Experiment with different embedding models
# - Add more documents to your knowledge base
# - Implement chat history for conversational RAG

print("🎉 RAG Tutorial Complete!")
print("You now understand the fundamentals of Retrieval-Augmented Generation!")
